# ⚓ Real-Time Webhook Receiver

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LineageLogic/LakeLogic/blob/main/examples/06_streaming/webhook/webhook_demo.ipynb) 
[![GitHub Repo](https://img.shields.io/badge/GitHub-Repo-blue?logo=github)](https://github.com/LineageLogic/LakeLogic/blob/main/examples/06_streaming/webhook/webhook_demo.ipynb)

## 🏢 Business Scenario

Modern enterprises increasingly rely on **event-driven architectures**. Instead of waiting for a nightly batch job to see what happened today, businesses want to respond to signals as they occur. Examples include:
*   **E-commerce**: A customer places an order on Stripe; you need to update inventory and trigger fulfillment instantly.
*   **SaaS/DevOps**: A new issue is opened on GitHub; you need to categorize it via AI and alert the engineering team.
*   **FinTech**: A high-value deposit is detected; you need to run fraud detection models before the funds are cleared.

## 💡 Value Proposition

*   **Zero-Latency Ingestion**: Eliminates the delay of polling. Your data lake remains perfectly in sync with your source APIs in real-time.
*   **Universal Connector**: Any system that can send a POST request (Stripe, Twilio, GitHub, Salesforce) can now feed your Lakehouse directly.
*   **Integrated Quality Gates**: Unlike simple message queues, LakeLogic validates every webhook event against a Data Contract *before* it touches your storage, preventing upstream application bugs from breaking your downstream analytics.
*   **Low Overhead**: You don't need to manage complex Kafka clusters for simple push-based ingestion. LakeLogic handles the micro-server setup automatically.

---

## 🎯 Goals

1. Define a Webhook source in a YAML contract
2. Start a local Webhook server using LakeLogic
3. Simulate incoming events (e.g., from GitHub or Stripe)
4. See real-time validation and processing

## 🚀 Step 1: Examine the Contract

Our contract `webhook_contract.yaml` defines a listener on port `8080` at the `/github-webhook` path.

In [ ]:
with open('webhook_contract.yaml', 'r') as f:
    print("📄 Webhook Contract Content:")
    print("-------------------------")
    print(f.read())

## ▶️ Step 2: Start the Webhook Listener

Since the processor runs in a blocking loop, we will start it in a background thread so we can send events to it from this same notebook.

In [ ]:
from lakelogic.core.streaming_processor import StreamingDataProcessor
import threading
import time

# Initialize processor
processor = StreamingDataProcessor(contract="webhook_contract.yaml", framework="bytewax")

# Start in a background thread
thread = threading.Thread(target=processor.start)
thread.daemon = True
thread.start()

print("🚀 Webhook receiver started in background!")
print("Listening at http://localhost:8080/github-webhook")
time.sleep(2) # Give it a moment to bind to the port

## 🧪 Step 3: Simulate Incoming Events

Now we'll use the `requests` library to send sample payloads to our running LakeLogic receiver.

In [ ]:
import requests
import json

url = "http://localhost:8080/github-webhook"

events = [
    {"action": "opened", "issue": {"title": "Feature Request: GraphQL support"}, "sender": {"login": "alice"}},
    {"action": "commented", "issue": {"title": "Fix SQL injection"}, "sender": {"login": "bob"}},
    {"action": "closed", "issue": {"title": "Update README"}, "sender": {"login": "charlie"}}
]

print("🧪 Sending test webhooks...")
for event in events:
    print(f"📤 Sending {event['action']} event for: {event['issue']['title']}")
    response = requests.post(url, json=event)
    print(f"📥 Server response: {response.status_code} - {response.json()}")
    time.sleep(1)

print("\n✅ Simulation complete!")

## 🛑 Step 4: Cleanup

Stop the background server.

In [ ]:
if 'processor' in locals():
    print("👋 Shutting down webhook server...")
    # Note: In a real notebook you might need to interrupt the kernel 
    # but here we'll try to shut down the server gracefully if implemented
    if hasattr(processor, 'stop'):
        processor.stop()

## 🎉 Summary

You just:
- ✅ Configured a push-based data source (Webhook)
- ✅ Launched a real-time HTTP receiver
- ✅ Successfully processed incoming event notifications

This pattern is ideal for event-driven architectures where your data pipeline needs to respond instantly to external triggers!